# Mitigating Catastrophic Forgetting in LLM Fine-Tuning using Experience Replay

This notebook explores catastrophic forgetting during supervised fine-tuning of a large language model.

The experiment uses **Mistral-7B-Instruct-v0.2** on selected MMLU subjects. The model is fine-tuned on Physics questions and evaluated on both Physics and Biology to observe whether performance on Biology drops after Physics-only fine-tuning.

To reduce forgetting, this notebook applies a simple **experience replay** strategy by mixing a small subset of Biology examples into the Physics fine-tuning dataset.

## Main Idea

When a model is fine-tuned on a new domain, it may improve on that domain but lose performance on another domain. This is known as **catastrophic forgetting**.

In this experiment:

- **Physics** is treated as the new fine-tuning domain.
- **Biology** is treated as the retained domain.
- A fixed subset of Biology examples is used as a replay buffer.
- The model is trained with Physics + Biology replay examples to reduce forgetting.

## Techniques Used

- Mistral-7B-Instruct-v0.2
- Supervised Fine-Tuning
- PEFT / LoRA
- Unsloth
- bitsandbytes
- MMLU benchmark
- Experience Replay

# 1. Environment Setup

This section installs the required libraries for efficient LLM fine-tuning.

The main libraries used are:

- **Unsloth**: efficient loading and fine-tuning of LLMs.
- **TRL**: provides `SFTTrainer` for supervised fine-tuning.
- **PEFT**: supports LoRA-based parameter-efficient fine-tuning.
- **bitsandbytes**: supports memory-efficient model loading and training utilities.
- **Accelerate**: helps manage model training on GPU hardware.

The notebook is intended to run in a GPU-enabled environment such as Google Colab.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

# 2. Imports and Experiment Configuration

This section imports the required Python libraries and defines the main experiment settings.

The experiment uses selected MMLU subjects grouped into two domains:

## Physics Domain

- `college_physics`
- `high_school_physics`
- `conceptual_physics`

## Biology Domain

- `college_biology`
- `high_school_biology`
- `medical_genetics`

The main hyperparameters include:

- `TRAIN_RATIO`: percentage of each subject used for training
- `REPLAY_RATIO`: size of the Biology replay subset relative to the Physics training set
- `NUM_EPOCHS`: number of fine-tuning epochs
- `BATCH_SIZE`: per-device training batch size
- `LR`: learning rate
- `MAX_LENGTH`: maximum sequence length

In [ ]:
import torch
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer

# ── Config ─────────────────────────────────────────────────────────
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

PHYSICS_SUBJECTS = [
    "college_physics",
    "high_school_physics",
    "conceptual_physics",
]
BIOLOGY_SUBJECTS = [
    "college_biology",
    "high_school_biology",
    "medical_genetics",
]

TRAIN_RATIO   = 0.80   # 80% of each subject for training
REPLAY_RATIO  = 0.25   # 25% biology mixed into physics during replay
MAX_LENGTH    = 512
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
NUM_EPOCHS = 3
LR         = 5e-5    # was 2e-4, 4x lower
BATCH_SIZE = 4
SEED          = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# 3. Output Directory

This section mounts Google Drive and creates a directory for saving experiment outputs.

The notebook saves intermediate and final results such as:

- `results.json`
- training outputs
- evaluation results
- final plots

Saving to Google Drive is useful because Colab sessions are temporary and may disconnect.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
SAVE_DIR = "/content/drive/MyDrive/Catastrophic_forgetting_unsloth"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

# 4. Building the MMLU Datasets

This section loads selected MMLU subjects and converts them into an instruction-style format suitable for Mistral fine-tuning.

Each MMLU example contains:

- a question
- four answer choices
- the correct answer index

The examples are formatted into a Mistral instruction prompt:

```text
<s>[INST] Question: ...
(A) option 1
(B) option 2
(C) option 3
(D) option 4
Answer with only the letter. [/INST] C</s>

In [ ]:
LETTERS = ["A", "B", "C", "D"]

def format_prompt(example):
    """
    Convert one MMLU example into Mistral instruction format.

    Input:
        question : "What is..."
        choices  : ["opt1", "opt2", "opt3", "opt4"]
        answer   : 2  (index, so correct answer is C)

    Output:
        text: "<s>[INST] Question: What is...
               (A) opt1
               (B) opt2
               (C) opt3
               (D) opt4
               Answer with only the letter. [/INST] C</s>"
    """
    choices_str = "\n".join(
        f"({LETTERS[i]}) {example['choices'][i]}"
        for i in range(4)
    )
    text = (
        f"<s>[INST] Question: {example['question']}\n"
        f"{choices_str}\n"
        f"Answer with only the letter. [/INST] "
        f"{LETTERS[example['answer']]}</s>"
    )
    return {"text": text}


def build_datasets(subjects, train_ratio=TRAIN_RATIO, seed=SEED):
    """
    Load MMLU test split for each subject, format prompts,
    then split 80/20 into train and eval.

    Returns:
        train_ds : combined training dataset (pure subject, no leakage)
        eval_ds  : combined eval dataset (held out forever)
    """
    all_train = []
    all_eval  = []

    for subject in subjects:
        # Load only the test split — clean subject-specific questions
        ds = load_dataset("cais/mmlu", subject, split="test")
        ds = ds.map(format_prompt)

        # 80/20 split — reproducible with fixed seed
        split    = ds.train_test_split(
            test_size=1 - train_ratio,
            seed=seed
        )
        train_ds = split["train"]
        eval_ds  = split["test"]

        all_train.append(train_ds)
        all_eval.append(eval_ds)

        print(f"  {subject}: {len(train_ds)} train | {len(eval_ds)} eval")

    combined_train = concatenate_datasets(all_train).shuffle(seed=seed)
    combined_eval  = concatenate_datasets(all_eval)

    print(f"  → Total train: {len(combined_train)} | eval: {len(combined_eval)}\n")
    return combined_train, combined_eval


print("Building Physics datasets...")
physics_train, physics_eval = build_datasets(PHYSICS_SUBJECTS)

print("Building Biology datasets...")
biology_train, biology_eval = build_datasets(BIOLOGY_SUBJECTS)

# Save dataset sizes for report
dataset_info = {
    "physics_train" : len(physics_train),
    "physics_eval"  : len(physics_eval),
    "biology_train" : len(biology_train),
    "biology_eval"  : len(biology_eval),
}
print("Dataset summary:", dataset_info)



# 5. Loading the Base Model with Unsloth

This section loads the base model: `mistralai/Mistral-7B-Instruct-v0.2`

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="mistralai/Mistral-7B-Instruct-v0.2",
    max_seq_length=320,
    dtype=torch.bfloat16,
    load_in_4bit=False,
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")


# 6. Applying LoRA Adapters

This section applies LoRA-based parameter-efficient fine-tuning.

Instead of updating all parameters in the Mistral model, LoRA trains a small set of adapter weights. This makes fine-tuning much more memory-efficient.

LoRA is applied to attention and MLP projection layers:

- `q_proj`
- `k_proj`
- `v_proj`
- `o_proj`
- `gate_proj`
- `up_proj`
- `down_proj`

In this experiment, the base model remains mostly frozen while the LoRA adapter weights are updated during training.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=128,
    lora_alpha=256,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# 7. Evaluation Function

This section defines the evaluation function used for MMLU-style multiple-choice questions.

Instead of generating a full text response, the model is evaluated using logit-based answer selection.

For each question, the model receives the prompt and the evaluation function compares the logits for the answer options:

```text
A, B, C, D

In [ ]:
def evaluate(model, tokenizer, eval_dataset, label, max_samples=None):
    """
    Evaluate model on MMLU eval set using logit-based answer selection.

    For each question, looks at model confidence for tokens A/B/C/D
    and picks the highest — no generation needed.

    Args:
        model       : the current model (base or fine-tuned)
        tokenizer   : Mistral tokenizer
        eval_dataset: held-out eval set (never trained on)
        label       : string label for printing e.g. "Physics"
        max_samples : optional cap for speed during debugging

    Returns:
        accuracy (float), correct (int), total (int)
    """
    model.eval()

    # Get the token IDs for " A", " B", " C", " D"
    # Space before letter is important — matches tokenization
    answer_token_ids = [
        tokenizer(" A", add_special_tokens=False).input_ids[-1],
        tokenizer(" B", add_special_tokens=False).input_ids[-1],
        tokenizer(" C", add_special_tokens=False).input_ids[-1],
        tokenizer(" D", add_special_tokens=False).input_ids[-1],
    ]

    samples = eval_dataset
    if max_samples:
        samples = eval_dataset.select(range(min(max_samples, len(eval_dataset))))

    correct = 0
    total   = 0

    with torch.no_grad():
        for example in samples:
            # Rebuild prompt WITHOUT the answer
            choices_str = "\n".join(
                f"({LETTERS[i]}) {example['choices'][i]}"
                for i in range(4)
            )
            prompt = (
                f"<s>[INST] Question: {example['question']}\n"
                f"{choices_str}\n"
                f"Answer with only the letter. [/INST]"
            )

            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH,
            ).to(DEVICE)

            outputs = model(**inputs)

            # Logits at the last token position — model predicts next token
            last_logits = outputs.logits[0, -1, :]

            # Extract scores for only A B C D
            abcd_logits = torch.tensor(
                [last_logits[tid] for tid in answer_token_ids]
            )
            predicted = abcd_logits.argmax().item()

            if predicted == example["answer"]:
                correct += 1
            total += 1

    accuracy = correct / total
    print(f"  {label}: {accuracy:.1%} ({correct}/{total})")
    return accuracy, correct, total



# 8. Supervised Fine-Tuning Function

This section defines the supervised fine-tuning function using TRL's `SFTTrainer`.

The trainer updates only the LoRA adapter parameters while using the formatted MMLU examples as supervised training data.

The training objective is standard language modeling loss over the instruction-formatted examples. In simple terms, the model learns to predict the correct answer letter given the question and answer choices.

In [ ]:
from trl import SFTTrainer, SFTConfig

def train_model(model, tokenizer, train_dataset, output_dir, epochs=NUM_EPOCHS):
    args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        learning_rate=LR,
        bf16=True,
        fp16=False,
        logging_steps=10,
        save_strategy="no",
        warmup_steps=20,
        lr_scheduler_type="cosine",
        report_to="none",
        packing=False,
    )
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        processing_class=tokenizer,
    )
    trainer.train()
    return model

# 9. Running the Continual Learning Experiment

The experiment is divided into three phases:

## Phase 1: Baseline Evaluation

Evaluate the base model on Physics and Biology before any fine-tuning.

## Phase 2: Physics-Only Fine-Tuning

Fine-tune the model only on Physics examples, then evaluate on both Physics and Biology.

This phase tests whether training on Physics causes the model to forget Biology.

## Phase 3: Experience Replay

Create a mixed training dataset using:

- all Physics training examples
- a small subset of Biology training examples

The Biology subset acts as a simple fixed replay buffer.

The model is then fine-tuned again on this mixed dataset and evaluated on both domains.

## Phase 1: Baseline Evaluation

In this phase, the base Mistral model is evaluated before any fine-tuning.

This provides the starting accuracy for:

- Physics
- Biology

These baseline scores are used to compare the effect of later fine-tuning.

In [ ]:
results = {}
print("=" * 55)
print("PHASE 1 — Baseline")
print("=" * 55)
p_acc, p_c, p_t = evaluate(model, tokenizer, physics_eval, "Physics")
b_acc, b_c, b_t = evaluate(model, tokenizer, biology_eval, "Biology")
results["Baseline"] = {"Physics": round(p_acc, 4), "Biology": round(b_acc, 4)}

with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Checkpoint saved.")

#

## Phase 2: Physics-Only Fine-Tuning

In this phase, the model is fine-tuned only on Physics examples.

After training, the model is evaluated again on both Physics and Biology.

If Biology accuracy drops after Physics-only fine-tuning, that indicates catastrophic forgetting.

In [ ]:
print("=" * 55)
print("PHASE 2 — Fine-tune on Physics only")
print("=" * 55)

model = train_model(
    model, tokenizer, physics_train,
    output_dir=f"{SAVE_DIR}/phase2_physics"
)

p_acc, p_c, p_t = evaluate(model, tokenizer, physics_eval, "Physics")
b_acc, b_c, b_t = evaluate(model, tokenizer, biology_eval, "Biology")
results["After Physics FT"] = {"Physics": round(p_acc, 4), "Biology": round(b_acc, 4)}

with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Checkpoint saved.")

## Phase 3: Experience Replay

In this phase, experience replay is used to reduce forgetting.

A subset of Biology examples is selected and mixed with the Physics training data.

The replay ratio is:

```python
REPLAY_RATIO = 0.25

In [ ]:
print("=" * 55)
print("PHASE 3 — Replay (Physics + 25% Biology)")
print("=" * 55)

n_bio      = int(len(physics_train) * REPLAY_RATIO)
bio_subset = biology_train.shuffle(seed=SEED).select(range(n_bio))
replay_ds  = concatenate_datasets([physics_train, bio_subset]).shuffle(seed=SEED)
print(f"Replay mix: {len(physics_train)} physics + {n_bio} biology = {len(replay_ds)} total")

model = train_model(
    model, tokenizer, replay_ds,
    output_dir=f"{SAVE_DIR}/phase3_replay"
)

p_acc, p_c, p_t = evaluate(model, tokenizer, physics_eval, "Physics")
b_acc, b_c, b_t = evaluate(model, tokenizer, biology_eval, "Biology")
results["After Replay"] = {"Physics": round(p_acc, 4), "Biology": round(b_acc, 4)}

with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)
print("All results saved.")
print(json.dumps(results, indent=2))


# 10. Visualizing Results

This section plots Physics and Biology accuracy across the three experiment phases:

1. Baseline
2. After Physics-only fine-tuning
3. After experience replay

The plot helps show whether Biology performance drops after Physics fine-tuning and whether experience replay recovers that performance.

In [ ]:
phases      = list(results.keys())
phys_scores = [results[p]["Physics"] * 100 for p in phases]
bio_scores  = [results[p]["Biology"]  * 100 for p in phases]

x = np.arange(len(phases))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - w/2, phys_scores, w, label="Physics", color="#4C72B0", edgecolor="white")
bars2 = ax.bar(x + w/2, bio_scores,  w, label="Biology",  color="#DD8452", edgecolor="white")

for bar, val in zip(bars1, phys_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
for bar, val in zip(bars2, bio_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(phases, fontsize=11)
ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_ylim(0, 100)
ax.set_title(
    "Catastrophic Forgetting & Experience Replay\nMistral-7B on MMLU (Physics vs Biology)",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/results_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to Google Drive.")

# 11. Results and Interpretation

The experiment compares model performance before and after fine-tuning.

| Phase | Physics Accuracy | Biology Accuracy |
|---|---:|---:|
| Baseline | 40.4% | 67.6% |
| After Physics Fine-Tuning | 41.4% | 59.5% |
| After Experience Replay | 52.5% | 66.7% |

## Interpretation

After Physics-only fine-tuning, Biology accuracy dropped from **67.6%** to **59.5%**. This shows catastrophic forgetting, where the model becomes more specialized toward the new domain and loses performance on the retained domain.

After applying experience replay, Biology accuracy recovered to **66.7%**, close to the baseline Biology performance. Physics accuracy also improved to **52.5%**.

This suggests that even a simple fixed replay buffer can help reduce catastrophic forgetting during supervised LLM fine-tuning.

## Key Takeaway

The main contribution of this notebook is not just fine-tuning an LLM. The main idea is testing a continual learning strategy:

> Mix a small number of old-domain examples into new-domain fine-tuning to reduce forgetting.

In this project:

- New domain: Physics
- Retained domain: Biology
- Replay buffer: fixed subset of Biology examples
- Fine-tuning method: supervised fine-tuning with LoRA adapters

# 12. Limitations and Future Work

This notebook is an experimental prototype for studying catastrophic forgetting in LLM fine-tuning.

## Limitations

- The replay buffer is simple and fixed.
- Only one replay ratio was tested.
- Biology replay examples were selected using a basic subset selection approach.
- The experiment uses selected MMLU subjects rather than a larger continual learning benchmark.
- The evaluation focuses only on accuracy.
- More rigorous train, validation, and test separation would improve the experimental design.

## Future Improvements

Future versions of this experiment could include:

- Testing multiple replay ratios such as 10%, 25%, and 50%.
- Randomly sampling Biology replay examples instead of selecting the first examples.
- Comparing replay with other continual learning methods.
- Adding per-subject accuracy analysis.
- Saving LoRA adapter checkpoints separately.
- Evaluating on more domains.
- Creating a reusable replay buffer abstraction.

